# Emerging Tech Lab - Concentration-Screen Opentrons Protocol

## Concentration-screen setup

In [ ]:
from conc_opentrons_helpers import (
    CHLOROFORM_TOP_UP_VOLUME_KEY,
    DIALDEHYDE_VOLUME_KEY,
    DIAMINE_VOLUME_KEY,
    log_step,
    log_absolute_time_tick,
    set_robot_speeds,
    validate_volume_for_p300,
    build_concentration_screen_plate_map,
    make_well_volume_map,
    make_condition_by_well_map,
    dispense_to_wells_with_tip_changes,
    summarise_concentration_screen_records,
)

import opentrons.execute

protocol = opentrons.execute.get_protocol_api("2.19")

# Loading labware
plate_48 = protocol.load_labware(
    "greenaway_48_wellplate_3750ul",
    location=2,
)
plate_8 = protocol.load_labware(
    "greenaway_8_wellplate_20000ul",
    location=3,
)
pip_rack = protocol.load_labware(
    "opentrons_96_tiprack_300ul",
    location=1,
)

# Load pipette
pip_300 = protocol.load_instrument(
    "p300_single_gen2",
    "left",  # change to "right" if needed
    tip_racks=[pip_rack],
)

# -----------------------------
# User setup: liquid handling
# -----------------------------

ASPIRATE_RATE = 70
STANDARD_DISPENSE_RATE = 70
SLOW_DISPENSE_RATE = 10  # used for dropwise dialdehyde addition
AIR_GAP_VOLUME = 15
MAX_DISPENSE = 200
PRE_WET_CYCLES = 3
PRE_WET_VOLUME = 180
TIP_CHANGE_INTERVAL = 3
TRANSFER_MODE = "fast"  # "fast" = one tip per reagent/source; "accurate" = change tips regularly

pip_300.flow_rate.aspirate = ASPIRATE_RATE
pip_300.flow_rate.dispense = STANDARD_DISPENSE_RATE

# Speed up robot movement while keeping liquid-handling flow rates controlled.
# Rim-touching during pre-wetting is slowed separately inside the helper function.
set_robot_speeds(
    protocol=protocol,
    pipette=pip_300,
    pipette_default_speed=400,
    max_head_speed=400,
)

# -----------------------------
# User setup: source locations
# -----------------------------

# 8-well source plate layout:
# A1: diamine stock
# B1: dialdehyde stock
# A2: chloroform placeholder/top-up solvent
DIAMINE_SOURCE = plate_8["A1"]
DIALDEHYDE_SOURCE = plate_8["B1"]
CHLOROFORM_SOURCE = plate_8["A2"]

# -----------------------------
# User setup: concentration conditions
# -----------------------------

# Default layout: each concentration condition occupies one column,
# with triplicates across rows A/B/C.
# Example: conc_1 -> A1, B1, C1; conc_2 -> A2, B2, C2.
# To upscale later, add more conditions, change REPLICATE_ROWS, change START_COLUMN,
# or set explicit target_wells for a condition.
REPLICATE_ROWS = ["A", "B", "C"]
START_COLUMN = 1

# Edit these placeholder volumes before running the real experiment.
# The placeholder values keep total liquid volume constant at 1000 uL per well.
CONCENTRATION_CONDITIONS = {
    "conc_1": {
        "label": "placeholder lowest concentration",
        "nominal_concentration": "placeholder_1",
        "column": 1,
        DIAMINE_VOLUME_KEY: 100,
        DIALDEHYDE_VOLUME_KEY: 100,
        CHLOROFORM_TOP_UP_VOLUME_KEY: 800,
    },
    "conc_2": {
        "label": "placeholder low concentration",
        "nominal_concentration": "placeholder_2",
        "column": 2,
        DIAMINE_VOLUME_KEY: 200,
        DIALDEHYDE_VOLUME_KEY: 200,
        CHLOROFORM_TOP_UP_VOLUME_KEY: 600,
    },
    "conc_3": {
        "label": "placeholder medium concentration",
        "nominal_concentration": "placeholder_3",
        "column": 3,
        DIAMINE_VOLUME_KEY: 300,
        DIALDEHYDE_VOLUME_KEY: 300,
        CHLOROFORM_TOP_UP_VOLUME_KEY: 400,
    },
    "conc_4": {
        "label": "placeholder high concentration",
        "nominal_concentration": "placeholder_4",
        "column": 4,
        DIAMINE_VOLUME_KEY: 400,
        DIALDEHYDE_VOLUME_KEY: 400,
        CHLOROFORM_TOP_UP_VOLUME_KEY: 200,
    },
}

# Select concentration condition(s) for this run.
# Add a new condition name here after adding it to CONCENTRATION_CONDITIONS.
CONDITIONS_TO_RUN = ["conc_1", "conc_2", "conc_3", "conc_4"]

if len(CONDITIONS_TO_RUN) == 0:
    raise ValueError("Select at least one concentration condition to run.")

## Basic OT-2 sanity test

In [ ]:
# -----------------------------
# Basic OT-2 sanity test
# -----------------------------

protocol.home()

log_step(protocol, "Starting basic OT-2 sanity test.")

# Test tip pickup/drop
pip_300.pick_up_tip()
log_step(protocol, "Picked up one tip successfully.")

# Test movement to source and target wells
pip_300.move_to(plate_8["A1"].top())
log_step(protocol, "Moved to source well A1 top.")

pip_300.move_to(plate_48["A1"].top())
log_step(protocol, "Moved to target well A1 top.")

pip_300.drop_tip()
log_step(protocol, "Dropped tip successfully.")

protocol.home()
log_step(protocol, "Basic OT-2 sanity test complete.")

## Automated concentration-screen execution

In [ ]:
# -----------------------------
# Build and validate the plate map
# -----------------------------

condition_plate_map = build_concentration_screen_plate_map(
    concentration_conditions=CONCENTRATION_CONDITIONS,
    conditions_to_run=CONDITIONS_TO_RUN,
    replicate_rows=REPLICATE_ROWS,
    start_column=START_COLUMN,
    protocol=protocol,
)

# Validate per-well volumes before moving the robot.
# Chloroform top-up can be zero for conditions that need no additional solvent.
for condition_name, condition in condition_plate_map.items():
    validate_volume_for_p300(
        condition[DIAMINE_VOLUME_KEY],
        f"Diamine ({condition_name})",
        max_single_dispense=MAX_DISPENSE,
        protocol=protocol,
    )
    validate_volume_for_p300(
        condition[DIALDEHYDE_VOLUME_KEY],
        f"Dialdehyde ({condition_name})",
        max_single_dispense=MAX_DISPENSE,
        protocol=protocol,
    )
    validate_volume_for_p300(
        condition[CHLOROFORM_TOP_UP_VOLUME_KEY],
        f"Chloroform top-up ({condition_name})",
        max_single_dispense=MAX_DISPENSE,
        allow_zero=True,
        protocol=protocol,
    )

condition_by_well = make_condition_by_well_map(condition_plate_map)
diamine_volumes_by_well = make_well_volume_map(condition_plate_map, DIAMINE_VOLUME_KEY)
chloroform_top_up_volumes_by_well = make_well_volume_map(condition_plate_map, CHLOROFORM_TOP_UP_VOLUME_KEY)
dialdehyde_volumes_by_well = make_well_volume_map(condition_plate_map, DIALDEHYDE_VOLUME_KEY)

# -----------------------------
# Automated concentration-screen setup
# Chemically sensible addition order:
#   1. diamine stock
#   2. chloroform top-up / dilution solvent
#   3. dialdehyde stock, slow/dropwise, to start the reaction
# -----------------------------

# Store dispense records for later inspection.
dispense_records_by_reagent = {}

# Add diamine stock to every selected concentration well first.
dispense_records_by_reagent["diamine"] = dispense_to_wells_with_tip_changes(
    pipette=pip_300,
    protocol=protocol,
    plate=plate_48,
    source_well=DIAMINE_SOURCE,
    target_well_volumes=diamine_volumes_by_well,
    dispense_rate=STANDARD_DISPENSE_RATE,
    reagent_name="diamine stock",
    tip_change_interval=TIP_CHANGE_INTERVAL,
    transfer_mode=TRANSFER_MODE,
    max_dispense=MAX_DISPENSE,
    air_gap_volume=AIR_GAP_VOLUME,
    pre_wet_cycles=PRE_WET_CYCLES,
    pre_wet_volume=PRE_WET_VOLUME,
    condition_by_well=condition_by_well,
    log_each_dispense_time=False,
    log_absolute_time=False,
)

# Add chloroform top-up before dialdehyde so each well has its intended dilution
# before the imine-forming reaction starts.
dispense_records_by_reagent["chloroform_top_up"] = dispense_to_wells_with_tip_changes(
    pipette=pip_300,
    protocol=protocol,
    plate=plate_48,
    source_well=CHLOROFORM_SOURCE,
    target_well_volumes=chloroform_top_up_volumes_by_well,
    dispense_rate=STANDARD_DISPENSE_RATE,
    reagent_name="chloroform top-up",
    tip_change_interval=TIP_CHANGE_INTERVAL,
    transfer_mode=TRANSFER_MODE,
    max_dispense=MAX_DISPENSE,
    air_gap_volume=AIR_GAP_VOLUME,
    pre_wet_cycles=PRE_WET_CYCLES,
    pre_wet_volume=PRE_WET_VOLUME,
    condition_by_well=condition_by_well,
    log_each_dispense_time=False,
    log_absolute_time=False,
)

# Add dialdehyde slowly/dropwise. This is the practical reaction-start event,
# so absolute timestamps are recorded for each target well.
log_absolute_time_tick(protocol, "Starting dialdehyde addition for concentration screen.")

dispense_records_by_reagent["dialdehyde"] = dispense_to_wells_with_tip_changes(
    pipette=pip_300,
    protocol=protocol,
    plate=plate_48,
    source_well=DIALDEHYDE_SOURCE,
    target_well_volumes=dialdehyde_volumes_by_well,
    dispense_rate=SLOW_DISPENSE_RATE,
    reagent_name="dialdehyde stock",
    tip_change_interval=TIP_CHANGE_INTERVAL,
    transfer_mode=TRANSFER_MODE,
    max_dispense=MAX_DISPENSE,
    air_gap_volume=AIR_GAP_VOLUME,
    pre_wet_cycles=PRE_WET_CYCLES,
    pre_wet_volume=PRE_WET_VOLUME,
    condition_by_well=condition_by_well,
    log_each_dispense_time=True,
    log_absolute_time=True,
)

log_absolute_time_tick(protocol, "Finished dialdehyde addition for concentration screen.")

# Reset dispense rate and summarise the run.
pip_300.flow_rate.dispense = STANDARD_DISPENSE_RATE

log_step(protocol, "Concentration-screen summary:")
concentration_screen_summary = summarise_concentration_screen_records(
    plate_map=condition_plate_map,
    dispense_records_by_reagent=dispense_records_by_reagent,
    protocol=protocol,
)

protocol.home()
